<a id="bayes-classifier"></a>

$$
\text{贝叶斯分类器 (Bayes Classifier)} \\ \quad \\
\text{由贝叶斯公式, 样本 x 属于类别 c 的后验概率:} \\
P(c \mid x) = \frac{p(x \mid c) P(c)}{p(x)} = \frac{p(x \mid c) P(c)}{\sum_{c'} p(x \mid c') P(c')} \\
\text{分母 p(x) 对所有类别相同, 所以分类只需要比较分子:} \\
\hat{c} = \arg\max_c P(c \mid x) = \arg\max_c \ p(x \mid c) P(c) \\ \quad \\
\text{为什么选后验最大的类别: 预测为 c 时, 预测正确的概率就是 } P(c \mid x) \\
\text{每个 x 都选让 } P(\text{正确} \mid x) \text{ 最大的类别, 总的错误率 } \int (1 - P(\hat{c} \mid x)) p(x) dx \text{ 就最小} \\ \quad \\
\text{难点: 似然 } p(x \mid c) \text{ 是 d 维的联合分布, 维度高时需要估计的参数太多}
$$

数学基础: [贝叶斯公式与全概率公式](../math_basics/statistics.ipynb#bayes) · 对比: [Sigmoid 的贝叶斯来源](sigmoid_function.ipynb#log-odds)

<a id="naive-assumption"></a>

$$
\text{朴素 (Naive) 假设: 给定类别后, 各个特征条件独立 (Conditional Independence)} \\
p(x \mid c) = p(x_1, x_2, \cdots, x_d \mid c) = \prod_{j=1}^d p(x_j \mid c) \\
\text{(这个假设通常不成立, 例如身高和体重相关, 但分类时往往仍然有效: 只需要后验最大的类别对, 不需要概率完全准确)} \\ \quad \\
\text{参数个数对比 (高斯情形, 每个类别):} \\
\text{完整的多元正态: 均值 d 个 + 协方差矩阵 } \frac{d(d+1)}{2} \text{ 个 (对称)} \\
\text{朴素贝叶斯: 均值 d 个 + 方差 d 个 (协方差矩阵被假设成对角矩阵)} \\
\text{d = 100 时: 5150 个 vs 200 个}
$$

<a id="gaussian-nb"></a>

$$
\text{高斯朴素贝叶斯 (Gaussian Naive Bayes): } p(x_j \mid c) = \frac{1}{\sqrt{2\pi\sigma_{jc}^2}} e^{-\frac{(x_j - \mu_{jc})^2}{2\sigma_{jc}^2}} \\ \quad \\
\text{参数用最大似然估计. 对数似然 (类别 c 有 } n_c \text{ 个样本):} \\
\ell = \sum_i \ln P(c_i) + \sum_i \sum_j \ln p(x_{ij} \mid c_i) \\
\text{每一项只和自己的参数有关, 可以分开最大化:} \\ \quad \\
\text{1. 均值和方差: 就是正态分布 MLE 的闭式解, 只用类别 c 的样本} \\
\mu_{jc} = \frac{1}{n_c} \sum_{i: c_i = c} x_{ij}, \quad \sigma_{jc}^2 = \frac{1}{n_c} \sum_{i: c_i = c} (x_{ij} - \mu_{jc})^2 \quad (\text{MLE 的方差除以 } n_c \text{, 不是 } n_c - 1) \\ \quad \\
\text{2. 先验 } \pi_c = P(c) \text{: } \max \sum_c n_c \ln \pi_c \ \text{ s.t. } \sum_c \pi_c = 1 \\
L = \sum_c n_c \ln \pi_c - \lambda \left(\sum_c \pi_c - 1\right), \quad \frac{\partial L}{\partial \pi_c} = \frac{n_c}{\pi_c} - \lambda = 0 \implies \pi_c = \frac{n_c}{\lambda} \\
\sum_c \pi_c = 1 \implies \lambda = \sum_c n_c = n \implies \pi_c = \frac{n_c}{n} \quad (\text{类别的频率}) \\ \quad \\
\text{sklearn 会给所有方差加上一个很小的平滑项 } \epsilon = 10^{-9} \times \max_j Var(x_j) \text{ (防止方差为 0 时除以 0)}
$$

数学基础: [正态分布参数的 MLE](../math_basics/statistics.ipynb#mle-normal) · [拉格朗日乘子法](../math_basics/calculus.ipynb#lagrange) · [伯努利分布的 MLE](../math_basics/statistics.ipynb#mle-bernoulli)

<a id="log-space"></a>

$$
\text{在对数空间计算 (Log Space)} \\
\text{d 个概率密度相乘, 每个都可能很小, 乘积会下溢 (underflow) 成 0, 所有类别都变成 0/0} \\
\ln \left[ P(c) \prod_j p(x_j \mid c) \right] = \ln \pi_c + \sum_j \left[ -\frac{1}{2} \ln (2\pi\sigma_{jc}^2) - \frac{(x_j - \mu_{jc})^2}{2\sigma_{jc}^2} \right] = a_c \\
\text{分类: } \arg\max_c a_c \text{ (ln 单调)} \\
\text{后验概率: } P(c \mid x) = \frac{e^{a_c}}{\sum_{c'} e^{a_{c'}}} = \text{softmax}(a)_c \text{, 用减去最大值的技巧计算}
$$

数学基础: [Softmax 的数值稳定性与 log-sum-exp](softmax_regression.ipynb#softmax-stability)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import logsumexp
from sklearn.datasets import load_wine
from sklearn.naive_bayes import GaussianNB

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

class GaussianNBManual:
    def fit(self, X, y, var_smoothing=1e-9):
        self.classes = np.unique(y)
        self.prior = np.array([np.mean(y == c) for c in self.classes])             # n_c / n
        self.mu = np.array([X[y == c].mean(axis=0) for c in self.classes])         # 类内均值
        eps = var_smoothing * X.var(axis=0).max()
        self.var = np.array([X[y == c].var(axis=0) for c in self.classes]) + eps   # MLE 方差 (ddof = 0) + 平滑
        return self

    def joint_log_likelihood(self, X):
        # a_c = ln pi_c + sum_j ln N(x_j | mu_jc, var_jc), 结果 n x K
        log_pdf = -0.5 * np.log(2 * np.pi * self.var)[None, :, :] - (X[:, None, :] - self.mu[None, :, :])**2 / (2 * self.var[None, :, :])
        return np.log(self.prior)[None, :] + log_pdf.sum(axis=2)

    def predict_proba(self, X):
        a = self.joint_log_likelihood(X)
        a = a - a.max(axis=1, keepdims=True)
        e = np.exp(a)
        return e / e.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes[np.argmax(self.joint_log_likelihood(X), axis=1)]

wine = load_wine()
X, y = wine.data, wine.target
rng = np.random.default_rng(0)
perm = rng.permutation(len(y))
X_train, y_train, X_test, y_test = X[perm[:120]], y[perm[:120]], X[perm[120:]], y[perm[120:]]
print(f'数据: {X.shape[0]} 个样本, {X.shape[1]} 个特征, {len(np.unique(y))} 个类别')

nb = GaussianNBManual().fit(X_train, y_train)
sk = GaussianNB().fit(X_train, y_train)
print(f'先验 == sklearn class_prior_ ? {np.allclose(nb.prior, sk.class_prior_)}, 先验 = {np.round(nb.prior, 4)}')
print(f'均值 == sklearn theta_ ? {np.allclose(nb.mu, sk.theta_)}')
print(f'方差 == sklearn var_ ? {np.allclose(nb.var, sk.var_)}')
print(f'predict_proba 一致? {np.allclose(nb.predict_proba(X_test), sk.predict_proba(X_test))}')
print(f'测试集准确率: 手算 = {np.mean(nb.predict(X_test) == y_test):.4f}, sklearn = {sk.score(X_test, y_test):.4f}')

# 直接在概率空间相乘 vs 对数空间
x0 = X_test[0]
pdf_each = np.exp(-0.5 * np.log(2 * np.pi * nb.var) - (x0 - nb.mu)**2 / (2 * nb.var))   # K x d
direct = nb.prior * pdf_each.prod(axis=1)
sci = lambda v: '[' + ', '.join(f'{t:.3e}' for t in v) + ']'
print(f'一个样本: 概率空间直接相乘 = {sci(direct)}, 对数空间 a_c = {nb.joint_log_likelihood(x0[None])[0]}')
print(f'后验: 概率空间归一化 = {direct / direct.sum()}, 对数空间 softmax = {nb.predict_proba(x0[None])[0]}')

# 特征很多时会下溢: 把 13 个特征重复 100 次 (1300 个特征)
X_rep = np.tile(X_train, 100)
nb_rep = GaussianNBManual().fit(X_rep, y_train)
x_rep = np.tile(x0, 100)
pdf_rep = np.exp(-0.5 * np.log(2 * np.pi * nb_rep.var) - (x_rep - nb_rep.mu)**2 / (2 * nb_rep.var))
print(f'1300 个特征直接相乘 = {sci(nb_rep.prior * pdf_rep.prod(axis=1))} (下溢成 0), 对数空间 a_c = {nb_rep.joint_log_likelihood(x_rep[None])[0]}')
print(f'log-sum-exp 归一化常数: 手算 = {np.max(nb.joint_log_likelihood(x0[None])) + np.log(np.sum(np.exp(nb.joint_log_likelihood(x0[None]) - np.max(nb.joint_log_likelihood(x0[None]))))):.6f}, scipy = {logsumexp(nb.joint_log_likelihood(x0[None])):.6f}')

<a id="nb-boundary"></a>

$$
\text{决策边界的形状: 两个类别的边界是 } a_1 = a_2 \\
a_1 - a_2 = \sum_j \left[ -\frac{(x_j - \mu_{j1})^2}{2\sigma_{j1}^2} + \frac{(x_j - \mu_{j2})^2}{2\sigma_{j2}^2} \right] + \text{const} \\
\text{各类方差不同时, } x_j^2 \text{ 的系数 } -\frac{1}{2\sigma_{j1}^2} + \frac{1}{2\sigma_{j2}^2} \neq 0 \text{, 边界是二次曲线} \\
\text{各类方差相同时, 二次项抵消, 边界是直线 (和逻辑回归的形式相同, 见 sigmoid 那一节的推导)}
$$

In [ ]:
# 二维例子: 两个类别的方差不同 -> 二次曲线边界
rng = np.random.default_rng(2)
Xa = rng.normal([0, 0], [1.0, 0.6], size=(200, 2))
Xb = rng.normal([1.5, 1.0], [0.4, 1.5], size=(200, 2))
X2 = np.vstack([Xa, Xb])
y2 = np.r_[np.zeros(200, dtype=int), np.ones(200, dtype=int)]
nb2 = GaussianNBManual().fit(X2, y2)
sk2 = GaussianNB().fit(X2, y2)

g1, g2 = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 5, 200))
grid = np.column_stack([g1.ravel(), g2.ravel()])
prob = nb2.predict_proba(grid)[:, 1].reshape(g1.shape)
print(f'网格上的后验和 sklearn 一致? {np.allclose(nb2.predict_proba(grid), sk2.predict_proba(grid))}')

plt.figure(figsize=(4, 3))
plt.contourf(g1, g2, prob, levels=np.linspace(0, 1, 11), cmap='RdBu_r', alpha=0.4)
plt.contour(g1, g2, prob, levels=[0.5], colors='k')
plt.scatter(*Xa.T, s=5, c='b', label='class 0')
plt.scatter(*Xb.T, s=5, c='r', label='class 1')
plt.title('quadratic boundary P(1|x) = 0.5')
plot_nicely()
plt.show()